# Phase 1 — EDA & Schema Map (v2, schema-aware)

**Owner:** Member 1 (Data & EDA Lead)  
**Goal:** Inventory every CSV, validate keys, lock the schema, surface red flags, publish the join graph.  
**Time budget:** 60–90 min.  
**Output:** `eda_insights.md` + green-light to start Phase 2.

## Key facts about this dataset

Unlike a typical hackathon dataset, Smadex has done a **lot** of the work for us:

- `creatives.csv` already contains hand-labeled visual features (`dominant_color`, `text_density`, `clutter_score`, `faces_count`, `has_discount_badge`, etc.) — no CV needed.
- `creative_summary.csv` already contains pre-computed fatigue metrics (`first_7d_ctr`, `last_7d_ctr`, `ctr_decay_pct`, `fatigue_day`) and a `creative_status` label (top_performer / stable / fatigued / underperformer).
- `creative_daily_country_os_stats.csv` is the time-series at creative × date × country × OS granularity.

**Strategic implication:** we don't *use* `creative_status` as our answer — we recompute fatigue ourselves and use the label as a sanity check. The pitch story is "our score is more actionable than the binary label."

In [17]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

DATA_DIR = Path("")
assert DATA_DIR.exists(), f"Drop the Smadex CSVs into {DATA_DIR.resolve()}"

EXPECTED = [
    "advertisers",
    "campaigns",
    "creatives",
    "creative_summary",
    "creative_daily_country_os_stats",
    "campaign_summary",
]

found = {p.stem: p for p in DATA_DIR.glob("*.csv")}
missing = [n for n in EXPECTED if n not in found]
extra   = [n for n in found if n not in EXPECTED]

print(f"Found {len(found)} CSVs.")
if missing: print(f"\u26a0  MISSING (expected from schema): {missing}")
if extra:   print(f"\u2139  EXTRA (not in schema, investigate): {extra}")
if not missing and not extra: print("\u2713  Files match the published schema exactly.")

Found 7 CSVs.
ℹ  EXTRA (not in schema, investigate): ['data_dictionary']


## 1. Load with the right dtypes

We know the schema, so we cast deliberately. IDs as nullable `Int64` to avoid float-precision issues.

In [18]:
ID_COLS = {"advertiser_id", "campaign_id", "creative_id"}
DATE_COLS = {"date", "start_date", "end_date", "creative_launch_date"}

def smart_load(path):
    df = pd.read_csv(path, low_memory=False)
    for c in df.columns:
        if c in ID_COLS:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
        elif c in DATE_COLS:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df

tables = {name: smart_load(found[name]) for name in EXPECTED if name in found}

for name, df in tables.items():
    print(f"  {name:<35} {len(df):>10,} rows \u00d7 {df.shape[1]:>3} cols")

  advertisers                                 36 rows ×   4 cols
  campaigns                                  180 rows ×  14 cols
  creatives                                1,080 rows ×  32 cols
  creative_summary                         1,080 rows ×  59 cols
  creative_daily_country_os_stats        192,315 rows ×  14 cols
  campaign_summary                           180 rows ×  22 cols


## 2. Lock the schema (single source of truth)

Hard-coded based on the published data dictionary. Members 2/3/4 reference this.

In [19]:
SCHEMA = {
    "advertisers": {
        "pk": "advertiser_id",
        "fks": [],
        "dimensions": ["vertical", "hq_region"],
    },
    "campaigns": {
        "pk": "campaign_id",
        "fks": ["advertiser_id"],
        "dates": ["start_date", "end_date"],
        "dimensions": ["vertical", "objective", "primary_theme",
                       "target_age_segment", "target_os", "kpi_goal"],
    },
    "creatives": {
        "pk": "creative_id",
        "fks": ["campaign_id"],
        "dates": ["creative_launch_date"],
        "dimensions": ["vertical", "format", "language", "theme", "hook_type",
                       "cta_text", "dominant_color", "emotional_tone"],
        "visual_features": ["text_density", "readability_score", "brand_visibility_score",
                            "clutter_score", "novelty_score", "motion_score",
                            "faces_count", "product_count", "duration_sec",
                            "copy_length_chars", "width", "height"],
        "booleans": ["has_price", "has_discount_badge", "has_gameplay", "has_ugc_style"],
        "asset_path": "asset_file",
    },
    "creative_summary": {
        "pk": "creative_id",
        "fks": ["campaign_id"],
        "label": "creative_status",
        "lifetime_metrics": ["total_spend_usd", "total_impressions", "total_clicks",
                             "total_conversions", "total_revenue_usd",
                             "overall_ctr", "overall_cvr", "overall_ipm", "overall_roas",
                             "perf_score"],
        "fatigue_metrics": ["first_7d_ctr", "last_7d_ctr", "ctr_decay_pct",
                            "first_7d_cvr", "last_7d_cvr", "cvr_decay_pct",
                            "peak_rolling_ctr_5", "fatigue_day", "total_days_active"],
    },
    "creative_daily_country_os_stats": {
        "pk": ["creative_id", "date", "country", "os"],
        "fks": ["campaign_id", "creative_id"],
        "dates": ["date"],
        "dimensions": ["country", "os"],
        "metrics": ["impressions", "viewable_impressions", "clicks", "conversions",
                    "spend_usd", "revenue_usd", "video_completions",
                    "impressions_last_7d", "days_since_launch"],
    },
    "campaign_summary": {
        "pk": "campaign_id",
        "fks": ["advertiser_id"],
        "lifetime_metrics": ["total_spend_usd", "total_impressions", "total_clicks",
                             "total_conversions", "total_revenue_usd",
                             "overall_ctr", "overall_cvr", "overall_roas"],
    },
}
print("Schema locked. Source of truth for the rest of the team.")

Schema locked. Source of truth for the rest of the team.


## 3. Verify primary keys are unique

In [20]:
print("Primary key uniqueness:\n")
for name, info in SCHEMA.items():
    if name not in tables: continue
    pk = info["pk"]
    df = tables[name]
    keys = pk if isinstance(pk, list) else [pk]
    dups = df.duplicated(subset=keys).sum()
    nulls = df[keys].isnull().any(axis=1).sum()
    status = "\u2713" if dups == 0 and nulls == 0 else "\u2717"
    print(f"  {status}  {name:<35} pk={keys}  dups={dups}  null_keys={nulls}")

Primary key uniqueness:

  ✓  advertisers                         pk=['advertiser_id']  dups=0  null_keys=0
  ✓  campaigns                           pk=['campaign_id']  dups=0  null_keys=0
  ✓  creatives                           pk=['creative_id']  dups=0  null_keys=0
  ✓  creative_summary                    pk=['creative_id']  dups=0  null_keys=0
  ✓  creative_daily_country_os_stats     pk=['creative_id', 'date', 'country', 'os']  dups=0  null_keys=0
  ✓  campaign_summary                    pk=['campaign_id']  dups=0  null_keys=0


## 4. Foreign-key referential integrity

Every FK should resolve to a PK in the parent table. Orphans = silent row loss in the master join.

In [21]:
pk_sets = {}
for name, info in SCHEMA.items():
    if name in tables and isinstance(info["pk"], str):
        pk_sets[info["pk"]] = (name, set(tables[name][info["pk"]].dropna()))

print("FK referential integrity:\n")
for name, info in SCHEMA.items():
    if name not in tables: continue
    df = tables[name]
    for fk in info.get("fks", []):
        if fk not in df.columns or fk not in pk_sets: continue
        target_table, target_set = pk_sets[fk]
        if target_table == name: continue
        fk_set = set(df[fk].dropna())
        orphans = fk_set - target_set
        n_rows = df[fk].isin(orphans).sum() if orphans else 0
        status = "\u2713" if not orphans else "\u26a0"
        print(f"  {status}  {name}.{fk} \u2192 {target_table}.{fk}: {len(orphans)} orphan keys, {n_rows} rows affected")

FK referential integrity:

  ✓  campaigns.advertiser_id → advertisers.advertiser_id: 0 orphan keys, 0 rows affected
  ✓  creatives.campaign_id → campaign_summary.campaign_id: 0 orphan keys, 0 rows affected
  ✓  creative_summary.campaign_id → campaign_summary.campaign_id: 0 orphan keys, 0 rows affected
  ✓  creative_daily_country_os_stats.campaign_id → campaign_summary.campaign_id: 0 orphan keys, 0 rows affected
  ✓  creative_daily_country_os_stats.creative_id → creative_summary.creative_id: 0 orphan keys, 0 rows affected
  ✓  campaign_summary.advertiser_id → advertisers.advertiser_id: 0 orphan keys, 0 rows affected


## 5. Date sanity

In [22]:
for name, info in SCHEMA.items():
    if name not in tables: continue
    df = tables[name]
    for d in info.get("dates", []):
        if d not in df.columns: continue
        s = df[d].dropna()
        if len(s) == 0:
            print(f"  \u26a0  {name}.{d}: all null")
            continue
        n_failed = df[d].isna().sum()
        print(f"  {name}.{d:<22} {s.min().date()} \u2192 {s.max().date()}  "
              f"({(s.max() - s.min()).days}d span, {n_failed} parse failures)")

if "campaigns" in tables:
    c = tables["campaigns"]
    bad = ((c["end_date"] < c["start_date"]).fillna(False)).sum()
    print(f"\n  Campaigns with end_date < start_date: {bad}")

  campaigns.start_date             2026-01-01 → 2026-01-21  (20d span, 0 parse failures)
  campaigns.end_date               2026-02-14 → 2026-04-05  (50d span, 0 parse failures)
  creatives.creative_launch_date   2026-01-01 → 2026-02-08  (38d span, 0 parse failures)
  creative_daily_country_os_stats.date                   2026-01-01 → 2026-03-16  (74d span, 0 parse failures)

  Campaigns with end_date < start_date: 0


## 6. Time-series density per creative

Critical for Member 2's fatigue model. If most creatives have <14 days, shift to a simpler decay heuristic.

In [23]:
if "creative_daily_country_os_stats" in tables:
    daily = tables["creative_daily_country_os_stats"]
    days_per_creative = daily.groupby("creative_id")["date"].nunique()
    
    print(f"Daily-stats rows: {len(daily):,}")
    print(f"Distinct creatives: {daily['creative_id'].nunique():,}")
    print(f"Distinct dates: {daily['date'].nunique()}\n")
    print("Days-per-creative distribution:")
    print(days_per_creative.describe(percentiles=[.1, .25, .5, .75, .9, .95]).round(1))
    print(f"\n  <7 days:    {(days_per_creative < 7).sum():>6,}  ({(days_per_creative < 7).mean()*100:.1f}%)")
    print(f"  7\u201313 days:  {((days_per_creative >= 7) & (days_per_creative < 14)).sum():>6,}")
    print(f"  14+ days:   {(days_per_creative >= 14).sum():>6,}  ({(days_per_creative >= 14).mean()*100:.1f}%)")
    
    if "creative_summary" in tables:
        cs = tables["creative_summary"]
        merged = cs[["creative_id", "total_days_active"]].merge(
            days_per_creative.rename("days_in_daily"), on="creative_id", how="left")
        mismatch = (merged["total_days_active"] != merged["days_in_daily"]).sum()
        print(f"\n  Cross-check vs creative_summary.total_days_active:")
        print(f"    Mismatches: {mismatch:,}")

Daily-stats rows: 192,315
Distinct creatives: 1,080
Distinct dates: 75

Days-per-creative distribution:
count    1080.0
mean       48.5
std         8.9
min        28.0
10%        37.0
25%        42.0
50%        48.0
75%        55.0
90%        59.0
95%        64.0
max        74.0
Name: date, dtype: float64

  <7 days:         0  (0.0%)
  7–13 days:       0
  14+ days:    1,080  (100.0%)

  Cross-check vs creative_summary.total_days_active:
    Mismatches: 0


## 7. Asset existence check

Frontend will load these PNGs. Verify they resolve.

In [24]:
ASSET_BASE = None
if "creatives" in tables:
    cr = tables["creatives"]
    paths = cr["asset_file"].dropna()
    base_candidates = [DATA_DIR, DATA_DIR.parent, Path("."),
                       DATA_DIR / "assets", DATA_DIR / "images",
                       DATA_DIR / "creative_assets", DATA_DIR / "creatives"]
    sample = paths.sample(min(200, len(paths)), random_state=42)
    best_hit = 0
    for base in base_candidates:
        if not base.exists(): continue
        hits = sum(1 for p in sample if (base / p).exists())
        if hits > best_hit:
            best_hit, ASSET_BASE = hits, base
    
    if ASSET_BASE and best_hit > 0:
        print(f"\u2713  Asset base directory: {ASSET_BASE.resolve()}")
        print(f"   Hit rate (sample of {len(sample)}): {best_hit}/{len(sample)} = {best_hit/len(sample)*100:.1f}%")
        for p in sample.head(3):
            full = ASSET_BASE / p
            print(f"    {full}  {'\u2713' if full.exists() else '\u2717'}")
    else:
        print("\u26a0  Could not resolve asset paths. Sample paths:")
        for p in sample.head(5): print(f"    {p}")

✓  Asset base directory: /Users/ledafarouk/Downloads/Smadex_Creative_Intelligence_Dataset_FULL
   Hit rate (sample of 200): 200/200 = 100.0%
    assets/creative_500423.png  ✓
    assets/creative_500548.png  ✓
    assets/creative_500731.png  ✓


## 8. Suspicious values & impossible rows

In [25]:
def check_metrics(name, df):
    print(f"\n--- {name} ---")
    metric_cols = [c for c in ["impressions", "viewable_impressions", "clicks",
                                "conversions", "spend_usd", "revenue_usd",
                                "total_impressions", "total_clicks", "total_conversions",
                                "total_spend_usd", "total_revenue_usd"] if c in df.columns]
    for c in metric_cols:
        s = pd.to_numeric(df[c], errors="coerce")
        n_neg = (s < 0).sum()
        n_null = s.isna().sum()
        flag = " \u26a0" if n_neg > 0 else ""
        print(f"  {c:<22} negatives={n_neg:>5}  nulls={n_null:>5}  max={s.max():>14.2f}{flag}")
    pairs = [("clicks", "impressions"), ("conversions", "clicks"),
             ("viewable_impressions", "impressions")]
    for child, parent in pairs:
        if child in df.columns and parent in df.columns:
            c_s = pd.to_numeric(df[child], errors="coerce")
            p_s = pd.to_numeric(df[parent], errors="coerce")
            bad = ((c_s > p_s) & p_s.notna()).sum()
            if bad > 0:
                print(f"  \u26a0  {bad} rows where {child} > {parent}")

for name in ["creative_daily_country_os_stats", "creative_summary", "campaign_summary"]:
    if name in tables: check_metrics(name, tables[name])


--- creative_daily_country_os_stats ---
  impressions            negatives=    0  nulls=    0  max=    1756049.00
  viewable_impressions   negatives=    0  nulls=    0  max=    1356079.00
  clicks                 negatives=    0  nulls=    0  max=      16345.00
  conversions            negatives=    0  nulls=    0  max=       2912.00
  spend_usd              negatives=    0  nulls=    0  max=       7933.07
  revenue_usd            negatives=    0  nulls=    0  max=      73525.41

--- creative_summary ---
  total_impressions      negatives=    0  nulls=    0  max=   33914048.00
  total_clicks           negatives=    0  nulls=    0  max=      86493.00
  total_conversions      negatives=    0  nulls=    0  max=      15689.00
  total_spend_usd        negatives=    0  nulls=    0  max=     106633.22
  total_revenue_usd      negatives=    0  nulls=    0  max=     483545.52

--- campaign_summary ---
  total_impressions      negatives=    0  nulls=    0  max=  111748246.00
  total_clicks     

## 9. Feature audit (the part that matters for ML/insights)

Distributions of every visual/content feature. Member 2 needs this to know which features have signal; Member 3/4 needs it to pick which slices to display.

In [26]:
if "creative_summary" in tables:
    cs = tables["creative_summary"]
    print("=" * 80)
    print("CATEGORICAL DIMENSION COUNTS")
    print("=" * 80)
    cat_cols = ["vertical", "format", "language", "theme", "hook_type",
                "cta_text", "dominant_color", "emotional_tone", "creative_status"]
    for c in cat_cols:
        if c in cs.columns:
            vc = cs[c].value_counts(dropna=False)
            print(f"\n{c}  ({cs[c].nunique()} distinct)")
            print(vc.head(15).to_string())

CATEGORICAL DIMENSION COUNTS

vertical  (6 distinct)
vertical
gaming           180
ecommerce        180
fintech          180
travel           180
food_delivery    180
entertainment    180

format  (5 distinct)
format
interstitial      344
native            289
rewarded_video    200
banner            199
playable           48

language  (7 distinct)
language
en    355
es    223
ja    117
fr    117
it     94
de     88
pt     86

theme  (20 distinct)
theme
discount         130
ugc               82
testimonial       78
drama             58
fantasy           57
feature-focus     56
celebrity         53
competitive       51
product-focus     49
food-closeup      49
trust             49
reward            45
minimalist        45
combo             44
wanderlust        43

hook_type  (24 distinct)
hook_type
free shipping       61
free rewards        54
security            54
limited offer       53
late-night          53
exclusive           51
fast                51
trending            51
watch n

In [27]:
if "creative_summary" in tables:
    cs = tables["creative_summary"]
    print("=" * 80)
    print("VISUAL FEATURE DISTRIBUTIONS")
    print("=" * 80)
    visual = [c for c in ["text_density", "readability_score", "brand_visibility_score",
                          "clutter_score", "novelty_score", "motion_score",
                          "faces_count", "product_count", "duration_sec", "copy_length_chars"]
              if c in cs.columns]
    print(cs[visual].describe(percentiles=[.1, .5, .9]).round(2).T)
    
    print("\n" + "=" * 80)
    print("BINARY FEATURE PREVALENCE")
    print("=" * 80)
    bools = [c for c in ["has_price", "has_discount_badge", "has_gameplay", "has_ugc_style"] if c in cs.columns]
    for c in bools:
        rate = cs[c].mean() * 100
        print(f"  {c:<24} {rate:>5.1f}% of creatives")

VISUAL FEATURE DISTRIBUTIONS
                         count   mean    std    min    10%    50%    90%     max
text_density            1080.0   0.42   0.19   0.05   0.17   0.42   0.68    0.95
readability_score       1080.0   0.81   0.11   0.42   0.67   0.82   0.95    0.99
brand_visibility_score  1080.0   0.61   0.17   0.15   0.40   0.61   0.84    0.98
clutter_score           1080.0   0.34   0.15   0.02   0.13   0.34   0.53    0.85
novelty_score           1080.0   0.60   0.21   0.05   0.32   0.60   0.88    0.99
motion_score            1080.0   0.55   0.24   0.00   0.24   0.53   0.90    1.00
faces_count             1080.0   1.13   1.04   0.00   0.00   1.00   3.00    4.00
product_count           1080.0   1.49   1.01   0.00   0.00   1.00   3.00    4.00
duration_sec            1080.0   5.83   8.12   0.00   0.00   0.00  20.00   30.00
copy_length_chars       1080.0  52.84  18.67  12.00  29.00  52.00  77.00  117.00

BINARY FEATURE PREVALENCE
  has_price                 36.4% of creatives
  has_

## 10. Quick wins — feature × performance correlations

Insight starter pack for Phase 4. Anything that jumps out is a candidate hero insight.

In [28]:
if "creative_summary" in tables:
    cs = tables["creative_summary"]
    print("Mean metrics by creative_status (sanity check on the labels):\n")
    g = cs.groupby("creative_status").agg(
        n=("creative_id", "count"),
        ctr=("overall_ctr", "mean"),
        cvr=("overall_cvr", "mean"),
        roas=("overall_roas", "mean"),
        decay=("ctr_decay_pct", "mean"),
        perf_score=("perf_score", "mean"),
    ).round(4)
    print(g)

Mean metrics by creative_status (sanity check on the labels):

                   n     ctr     cvr    roas   decay  perf_score
creative_status                                                 
fatigued         199  0.0068  0.1262  3.6595 -0.8421      0.6679
stable           740  0.0050  0.0939  3.6628 -0.7768      0.4802
top_performer     46  0.0092  0.1014  7.4388 -0.6699      0.8401
underperformer    95  0.0034  0.0715  1.5304 -0.7344      0.1436


In [29]:
if "creative_summary" in tables:
    cs = tables["creative_summary"]
    MIN_IMPR = 50_000
    eligible = cs[cs["total_impressions"] >= MIN_IMPR].copy()
    print(f"Creatives with \u2265{MIN_IMPR:,} impressions: {len(eligible):,} of {len(cs):,}\n")
    
    findings = []
    dims = ["vertical", "format", "theme", "hook_type", "cta_text",
            "dominant_color", "emotional_tone"]
    for d in dims:
        if d not in eligible.columns: continue
        g = eligible.groupby(d).agg(n=("creative_id","count"),
                                     ctr=("overall_ctr","mean"),
                                     roas=("overall_roas","mean")).query("n >= 20")
        if len(g) < 2: continue
        ctr_spread = g["ctr"].max() / g["ctr"].min() if g["ctr"].min() > 0 else float("inf")
        findings.append((d, ctr_spread, g["ctr"].idxmax(), g["ctr"].idxmin(),
                         g.loc[g["ctr"].idxmax(), "ctr"], g.loc[g["ctr"].idxmin(), "ctr"]))
    
    findings.sort(key=lambda x: -x[1])
    print("Dimensions ranked by CTR spread (best/worst level):\n")
    for d, spread, top, bot, top_ctr, bot_ctr in findings:
        print(f"  {d:<20} {spread:>5.2f}x   best={top!r:<22} ({top_ctr:.4f})  "
              f"worst={bot!r:<22} ({bot_ctr:.4f})")

Creatives with ≥50,000 impressions: 1,080 of 1,080

Dimensions ranked by CTR spread (best/worst level):

  format                2.20x   best='playable'             (0.0086)  worst='banner'               (0.0039)
  vertical              1.87x   best='gaming'               (0.0074)  worst='fintech'              (0.0040)
  hook_type             1.86x   best='competition'          (0.0074)  worst='earn more'            (0.0040)
  theme                 1.66x   best='gameplay'             (0.0072)  worst='trust'                (0.0043)
  dominant_color        1.51x   best='lime'                 (0.0059)  worst='green'                (0.0039)
  emotional_tone        1.09x   best='adventurous'          (0.0056)  worst='trustworthy'          (0.0051)
  cta_text              1.04x   best='Learn more'           (0.0056)  worst='Order now'            (0.0053)


In [30]:
if "creative_summary" in tables:
    cs = tables["creative_summary"]
    eligible = cs[cs["total_impressions"] >= 50_000].copy()
    visual = [c for c in ["text_density", "readability_score", "brand_visibility_score",
                          "clutter_score", "novelty_score", "motion_score",
                          "faces_count", "product_count", "copy_length_chars",
                          "has_price", "has_discount_badge", "has_gameplay", "has_ugc_style"]
              if c in eligible.columns]
    
    corrs = eligible[visual + ["overall_ctr"]].corr()["overall_ctr"].drop("overall_ctr").sort_values()
    print("Pearson correlation with CTR (overall, no controls):\n")
    for feat, c in corrs.items():
        bar = "+" * int(abs(c) * 50) if c > 0 else "-" * int(abs(c) * 50)
        print(f"  {feat:<26} {c:>+.3f}  {bar}")
    print("\nNote: unconditioned. Member 2 should redo with vertical fixed-effects.")

Pearson correlation with CTR (overall, no controls):

  readability_score          -0.212  ----------
  brand_visibility_score     -0.145  -------
  product_count              -0.126  ------
  text_density               -0.099  ----
  copy_length_chars          -0.088  ----
  faces_count                -0.072  ---
  has_ugc_style              -0.071  ---
  has_price                  -0.070  ---
  has_discount_badge         -0.045  --
  clutter_score              +0.129  ++++++
  has_gameplay               +0.322  ++++++++++++++++
  novelty_score              +0.369  ++++++++++++++++++
  motion_score               +0.404  ++++++++++++++++++++

Note: unconditioned. Member 2 should redo with vertical fixed-effects.


## 11. Generate `eda_insights.md` (team handoff doc)

In [31]:
lines = []
lines.append("# Smadex EDA & Schema Map\n")
lines.append(f"_Generated: {datetime.now().isoformat(timespec='seconds')}_\n")
lines.append("_Source of truth for Members 2 (ML), 3 (frontend), 4 (insights). "
             "If you need to change a column convention, ping the team in chat first._\n\n")

lines.append("## Tables (row counts)\n")
for name, df in tables.items():
    lines.append(f"- `{name}.csv` \u2014 **{len(df):,}** rows \u00d7 {df.shape[1]} cols")
lines.append("")

lines.append("## Join graph\n")
lines.append("```")
lines.append("advertisers")
lines.append("    \u2514\u2500\u2500 campaigns           (FK: advertiser_id)")
lines.append("            \u2514\u2500\u2500 creatives             (FK: campaign_id)")
lines.append("                    \u251c\u2500\u2500 creative_summary       (PK: creative_id)")
lines.append("                    \u2514\u2500\u2500 creative_daily_country_os_stats   (composite PK: creative_id, date, country, os)")
lines.append("```\n")

lines.append("## Master table recipe (Phase 2)\n")
lines.append("`master_creatives.csv` = one row per creative_id, built from:\n")
lines.append("```")
lines.append("creative_summary  (spine \u2014 one row per creative_id, pre-computed metrics)")
lines.append("  \u27f5 creatives          ON creative_id  (visual features, asset_file)")
lines.append("  \u27f5 campaigns          ON campaign_id  (objective, primary_theme)")
lines.append("  \u27f5 advertisers        ON advertiser_id  (hq_region)")
lines.append("```\n")
lines.append("Use **left joins from `creative_summary`** so we never lose a creative. "
             "After every join, assert `len(master) == len(creative_summary)`.\n")

lines.append("## Daily fatigue table recipe (Phase 3)\n")
lines.append("`daily_fatigue_table.csv` = `creative_daily_country_os_stats` aggregated to creative \u00d7 date "
             "(collapse country/OS for the main fatigue signal). Keep the country/OS-level table too as "
             "`daily_fatigue_country_os.csv` for slice analysis.\n")

lines.append("## Conventions (locked)\n")
lines.append("- ID columns are `Int64` (nullable). Don't cast to float.")
lines.append("- Date columns are `pd.Timestamp` (UTC-naive).")
lines.append("- Spend/revenue use the `_usd` suffix.")
lines.append("- Metrics: `ctr = clicks/impressions`, `cvr = conversions/clicks`, "
             "`cpc = spend/clicks`, `cpa = spend/conversions`, `roas = revenue/spend`. "
             "Divide-by-zero \u2192 `NaN`, never `inf`.")
lines.append("- `creative_status` is the synthetic ground-truth label \u2014 **use only for validation**, "
             "not as a model input.")
asset_str = str(ASSET_BASE.resolve()) if ASSET_BASE else "TBD (asset path resolution failed in \u00a77)"
lines.append(f"- Asset files live under: `{asset_str}`\n")

lines.append("## Insights placeholder (Phase 4)\n")
lines.append("_Add 5 strong insights here. Each must include: number, sample size filter, and a one-line \"so what.\"_\n")

Path("eda_insights.md").write_text("\n".join(lines))
print("\u2713  Wrote eda_insights.md")
print("\n--- Preview ---\n")
print("\n".join(lines[:30]))

✓  Wrote eda_insights.md

--- Preview ---

# Smadex EDA & Schema Map

_Generated: 2026-04-25T11:52:40_

_Source of truth for Members 2 (ML), 3 (frontend), 4 (insights). If you need to change a column convention, ping the team in chat first._


## Tables (row counts)

- `advertisers.csv` — **36** rows × 4 cols
- `campaigns.csv` — **180** rows × 14 cols
- `creatives.csv` — **1,080** rows × 32 cols
- `creative_summary.csv` — **1,080** rows × 59 cols
- `creative_daily_country_os_stats.csv` — **192,315** rows × 14 cols
- `campaign_summary.csv` — **180** rows × 22 cols

## Join graph

```
advertisers
    └── campaigns           (FK: advertiser_id)
            └── creatives             (FK: campaign_id)
                    ├── creative_summary       (PK: creative_id)
                    └── creative_daily_country_os_stats   (composite PK: creative_id, date, country, os)
```

## Master table recipe (Phase 2)

`master_creatives.csv` = one row per creative_id, built from:

```
creative_summary  

## 12. Done-when checklist

- [ ] §3: every PK has 0 duplicates and 0 null keys
- [ ] §4: FK orphans are 0 (or noted in `eda_insights.md`)
- [ ] §5: dates parse cleanly, no end < start
- [ ] §6: at least 50% of creatives have ≥14 days of data — if not, flag Member 2
- [ ] §6: `total_days_active` matches the daily-table date count
- [ ] §7: asset files resolve to a real directory
- [ ] §8: no negative metrics, no clicks > impressions
- [ ] §10: at least 3 surprising findings dropped into `notes.md` for Phase 4
- [ ] `eda_insights.md` committed and pinned in team chat

**Green across the board → start Phase 2 (`02_master_table.ipynb`).**